Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

#In Python, the syntax from X import Y means X is the container (a package or a module) and Y is the object being pulled out of it (a sub-package, a module, a class, a function, or a variable)

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002802D698AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002802D699550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic._internal import _model_construction
from pydantic import BaseModel, Field

class Movie (BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating: float=Field(description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie)
model_with_structure #A RunnableBinding is a LangChain object that wraps an existing Runnable and permanently attaches configuration to it, without changing the original runnable.
#a RunnableBinding around a chat model) is simply a chat model with some settings already attached to it.It is is not a new LLM.It is a ChatModelBinding (a wrapper around the original chat model).
model.invoke("Tell about the movie Bahubali")




_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002802D698AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002802D699550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

The Items that we can bind :

temperature,
max_tokens,
tools,
structured output schema,
callbacks,
stop sequences,
other configuration

In [5]:
model_with_structure.invoke("Tell about the movie Bahubali")

Movie(title='Bahubali', year=2015, director='S.S. Rajamouli', rating=8.8)

#Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field
# importing two things from the Pydantic library.
# BaseModel → used to create a structured data model.
# Field → used to provide extra information about each field.

#Create a class called Movie that inherits all the functionality of BaseModel class. Now Movie automatically gets validation,serialization,JSON conversion,type checking
class Movie (BaseModel):
        """A movie with details."""
        title: str=Field(..., description="The title of the movie") #assigning a special Field object instead of a string. Ellipsis object means this field is required.
        year: int =Field(..., description="The year the movie was released")
        director: str=Field(..., description="The director of the movie")
        rating: float= Field(..., description="The movie's rating")

model_with_structure=model.with_structured_output(Movie,
include_raw=True)
response=model_with_structure.invoke("Movie bahubali") #just creating another model.,Don't return free-form text. Return data that matches the Movie schema.
response
#The parsed object is simply the structured Python object created after interpreting and validating the model's output.


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie "Bahubali." Let me see what I need to do here. The tools provided include a Movie function that requires title, year, director, and rating. The user didn\'t specify which part of Bahubali they want, but since it\'s a popular movie, I should probably provide the details using the Movie function.\n\nFirst, I need to recall the details of Bahubali. The original Bahubali movie was released in 2015, directed by S.S. Rajamouli. The title is "Bahubali: The Beginning." The rating is around 8.1 on IMDb. Wait, there\'s also a sequel, Bahubali 2, which came out in 2017. But the user just said "Movie bahubali," so maybe they\'re referring to the first one. I should check if there\'s any ambiguity here. However, since the user didn\'t specify, I\'ll go with the first one. \n\nI need to make sure all required parameters are included: title, year, director, and rating. Let me structure the 

Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor (BaseModel):
        name: str
        role: str

class MovieDetails (BaseModel):
        title:str
        year: int
        cast: list[Actor]
        genres: list[str]
        budget: float | None=Field (None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)